In [1]:
import tensorflow as tf
from tensorflow.keras.models import model_from_json

# load model
json = 'model_rnn_GRU.120_Dense.5010_LSTMKernelInit.VarianceScaling_DenseKernelInit.lecun_uniformKRl1.0_KRl2.0_recAct.sigmoid_arch.json'
with open(json, 'r') as json_file:
    load = json_file.read()
model = model_from_json(load)

# load weights
weights = 'model_rnn_GRU.120_Dense.5010_LSTMKernelInit.VarianceScaling_DenseKernelInit.lecun_uniformKRl1.0_KRl2.0_recAct.sigmoid_weights.h5'
model.load_weights(weights)

model.summary()

2026-04-01 20:08:13.479368: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-01 20:08:13.481652: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-01 20:08:13.522026: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-01 20:08:13.522046: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-01 20:08:13.523159: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 15, 6)]           0         
                                                                 
 gru (GRU)                   (None, 120)               46080     
                                                                 
 dense_0 (Dense)             (None, 50)                6050      
                                                                 
 dense_1 (Dense)             (None, 10)                510       
                                                                 
 output_softmax (Dense)      (None, 3)                 33        
                                                                 
Total params: 52673 (205.75 KB)
Trainable params: 52673 (205.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [5]:
# ===== Rebuild GRU with return_sequences=True =====
original_gru = model.get_layer('gru')

gru_seq = GRU(
    units=original_gru.units,
    return_sequences=True
)

x_input = model.input
x_output = gru_seq(x_input)

gru_model = Model(inputs=x_input, outputs=x_output)

# copy weights
gru_seq.set_weights(original_gru.get_weights())

# ===== Input: all -1 =====
x = np.full((1, 15, 6), -1.0)

# ===== Run =====
output = gru_model.predict(x)[0]  # shape: (15, 120)

# ===== Print outputs =====
for t in range(15):
    print(f"\nTimestep {t}:")
    print(output[t])

1/1 [==============================] - 0s 297ms/step

Timestep 0:
[-0.10595808  0.50266093 -0.09098496  0.37063414 -0.1900891   0.23851772
  0.36087286 -0.03923282  0.13058342 -0.17268197 -0.3816001  -0.08093067
  0.32054898  0.17100951 -0.51292866  0.48131785 -0.02688476  0.2760943
 -0.7692829  -0.20354176 -0.00636206  0.23657437  0.7753754   0.1007819
 -0.04238403  0.2525825  -0.05450045 -0.45656857 -0.7060123  -0.21112387
 -0.44580022 -0.14875725 -0.0651975  -0.03641514  0.2639181   0.01772167
 -0.15910766 -0.56025916  0.5449277  -0.29875717 -0.13504429 -0.67636275
 -0.07082765 -0.6551675   0.31305474  0.16013524  0.82701784 -0.778101
  0.43351993 -0.5753689   0.15079682 -0.4027368   0.00644975 -0.48217475
  0.27999333 -0.3364508   0.53906316  0.83284026 -0.67117095  0.31947744
  0.3200907  -0.4236306  -0.03982303 -0.0193469   0.80372226  0.6115832
  0.0965642  -0.6865527  -0.01788119  0.53773916  0.05119384 -0.34542853
 -0.09200399 -0.58913124  0.7796595  -0.6163545  -0.21828282 -0

In [ ]:
# Run this cell to import the neccessary imports.
# Make sure you have the following files in your working directory
#        "inputs.txt"
#        "relu_bias.txt"
#        "relu_weights.txt"
#        "sigmoid_bias.txt"
#        "sigmoid_weights.txt"

import numpy as np
import os

# input binaries
in_bin_folder = './input_binaries'
if not os.path.exists(in_bin_folder):
    os.makedirs(in_bin_folder)

# relu package sv file folder
relu_pkg_folder = './relu_weights_biases_pkgs'
if not os.path.exists(relu_pkg_folder):
    os.makedirs(relu_pkg_folder)

# sigmoid package sv file folder
sig_pkg_folder = './sigmoid_weights_biases_pkgs'
if not os.path.exists(sig_pkg_folder):
    os.makedirs(sig_pkg_folder)


# # gru package sv file folder
gru_pkg_folder = './weights/gru_weights_biases_pkgs'
if not os.path.exists(gru_pkg_folder):
    os.makedirs(gru_pkg_folder)

# # binary that goes to tanh bram/look up table
if not os.path.exists('./tanh_BRAM_binaries'):
    os.makedirs('./tanh_BRAM_binaries')

# # dense 0 package sv file folder
dense_0_pkg_folder = './weights/dense_0_weights_biases_pkgs'
if not os.path.exists(dense_0_pkg_folder):
    os.makedirs(dense_0_pkg_folder)

# # dense 1 package sv file folder
dense_1_pkg_folder = './weights/dense_1_weights_biases_pkgs'
if not os.path.exists(dense_1_pkg_folder):
    os.makedirs(dense_1_pkg_folder)



def flip_bits(val, bits):
    return ((val ^ (2**bits - 1)) + 1)

def sigmoid(x):
    return 1/(1+(np.e**(-x)))

# really innefficient but calling "sed" subprocess didn't work
# Removes negative sign "-" from file
def remove_negs(tmp_file, dest_file_name):
    output_file = open(tmp_file, 'r')
    dest_file = open(dest_file_name, 'w')
    file = output_file.read()
    file = file.replace('b-', 'b')
    dest_file.write(file)
    # close files
    dest_file.close()
    output_file.close()
    
# =========================================================================
# pkg functions (for sub-processes)
# =========================================================================
def get_pkg_name(name, width, nfrac):
    # MIGHT WANT TO CHANGE THIS FOR CLARITY
    return str(name) + '_' + str(width) + '_' + str(width - nfrac)

def conv_to_str(num, width, nfrac):

    # shift the decimal point to the right
    div = (2**nfrac)*1.0
    x = round(float(num * div // 1))

    if (x < 0):
        return format(flip_bits(x, width), '0'+ str(width) + 'b')
    return format(x, '0'+ str(width) + 'b')


# =========================================================================

# print(conv_to_str(4.127311706542969, 4, 2))
print(conv_to_str(-1.5, 4, 2))
# print(conv_to_str(-2.344774007797241, 10, 5))

-1010


In [ ]:
# Extract weights and biases for each layer (gru layer separated to 3 dense) and print the weights to output
# and to files with name layer_weight.txt and layer_bias.txt
for layer in model.layers:
    print(layer.name)
    if layer.name == 'gru':

        print(layer.get_config())

        weights = layer.get_weights()
        print("Number of weights tensors: " + str(len(weights)))

        # W for x, U for h, B for biases
        W = weights[0]
        U = weights[1]
        B = weights[2]

        print("input size " + str(W.shape[0]))
        print("hidden state size " + str(layer.units))

        # extract three weights and biases for the three dense layers

        reset_gate_weight = np.concatenate((W[:, :layer.units], U[:, :layer.units]), axis = 0)
        update_gate_weight = np.concatenate((W[:, layer.units:2*layer.units], U[:, layer.units:2*layer.units]), axis=0)
        candidate_gate_weight = np.concatenate((W[:, 2*layer.units:], U[:, 2*layer.units:]), axis=0)

        # combine input biases and recurrent biases
        # notice recurrent biases is used when reset_after config set to True for the gru layer (default is True)
        reset_gate_bias = np.sum(B[:, :layer.units], axis = 0)
        update_gate_bias = np.sum(B[:, layer.units:2*layer.units], axis = 0)
        candidate_gate_bias = np.sum(B[:, 2*layer.units:], axis = 0)

        print("reset_gate_weight")
        print(reset_gate_weight.shape)
        print("update_gate_weight")
        print(update_gate_weight.shape)
        print("candidate_gate_weight")
        print(candidate_gate_weight.shape)

        print("reset_gate_bias")
        print(reset_gate_bias.shape)
        print("update_gate_bias")
        print(update_gate_bias.shape)
        print("candidate_gate_bias")
        print(candidate_gate_bias.shape)


        # find maximum and minimum values for each weight and bias
        print("max and min values")
        print("reset_gate_weight")
        print(np.max(reset_gate_weight))
        print(np.min(reset_gate_weight))
        print("update_gate_weight")
        print(np.max(update_gate_weight))
        print(np.min(update_gate_weight))
        print("candidate_gate_weight")
        print(np.max(candidate_gate_weight))
        print(np.min(candidate_gate_weight))

        print("reset_gate_bias")
        print(np.max(reset_gate_bias))
        print(np.min(reset_gate_bias))
        print("update_gate_bias")
        print(np.max(update_gate_bias))
        print(np.min(update_gate_bias))
        print("candidate_gate_bias")
        print(np.max(candidate_gate_bias))
        print(np.min(candidate_gate_bias))
        

        # write to npy files
        np.savetxt('reset_gate_weights.txt', reset_gate_weight.flatten())
        np.savetxt('update_gate_weights.txt', update_gate_weight.flatten())
        np.savetxt('candidate_gate_weights.txt', candidate_gate_weight.flatten())

        np.savetxt('reset_gate_biases.txt', reset_gate_bias.flatten())
        np.savetxt('update_gate_biases.txt', update_gate_bias.flatten())
        np.savetxt('candidate_gate_biases.txt', candidate_gate_bias.flatten())


    elif layer.name != "input_1":
        # all other layers are dense with relu activation
        
        # print layer info (activation)
        print(layer.get_config())
        dense_weights = layer.get_weights()
        weights = dense_weights[0]
        biases = dense_weights[1]

        # print weights and biases
        print("weights shape")
        print(weights.shape)
        print("biases shape")
        print(biases.shape)

        # find maximum and minimum values for each weight and bias
        print("max and min values")
        print("weights")
        print(np.max(weights))
        print(np.min(weights))
        print("biases")
        print(np.max(biases))
        print(np.min(biases))    

        # # write to a txt file, flatten the weights
        np.savetxt(layer.name + '_weights.txt', weights.flatten())
        np.savetxt(layer.name + '_biases.txt', biases.flatten())
    
    print("=====================================================================================================")


# ALter this to reflect state of files
NUM_INPUTS = 10

INPUT_SIZE_DENSE_0 = 120
OUTPUT_SIZE_DENSE_0 = 50

INPUT_SIZE_DENSE_1 = 50
OUTPUT_SIZE_DENSE_1 = 10

INPUT_SIZE_OUT_DENSE = 10
OUTPUT_SIZE_OUT_DENSE = 3

INPUT_SIZE_GRU = 6
OUTPUT_SIZE_GRU = 120

input_1
gru
{'name': 'gru', 'trainable': True, 'dtype': 'float32', 'return_sequences': False, 'return_state': False, 'go_backwards': False, 'stateful': False, 'unroll': False, 'time_major': False, 'units': 120, 'activation': 'tanh', 'recurrent_activation': 'sigmoid', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'VarianceScaling', 'config': {'scale': 1.0, 'mode': 'fan_in', 'distribution': 'truncated_normal', 'seed': None}, 'registered_name': None}, 'recurrent_initializer': {'module': 'keras.initializers', 'class_name': 'Orthogonal', 'config': {'gain': 1.0, 'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'recurrent_regularizer': None, 'bias_regularizer': None, 'activity_regularizer': None, 'kernel_constraint': None, 'recurrent_constraint': None, 'bias_constraint': None, 'dropout': 0.0, 'recurrent_dropout': 0.0, 

In [5]:
# =======================================================
# generate dense wights and bias package
# =======================================================

def generate_dense_pkg_file(name, dest_folder, width, nfrac, weights_file, biases_file, input_size, output_size):
# arguments:
# name: name of the layer
# width: width of the binary
# nfrac: number of fractional bits
# weights_file: file containing weights
# biases_file: file containing biases
# num_weights: number of weights
# num_biases: number of biases
    
    # determine the maximum and minimum values the fix point number can represent
    max_val = (2**(width-1) - 1)/(2**nfrac)
    min_val = -2**(width-1)/(2**nfrac)
    print("Max value: " + str(max_val))
    print("Min value: " + str(min_val))

    # open source files
    src_weights_file = open(weights_file, 'r')
    src_biases_file = open(biases_file, 'r')
    
    package_name = get_pkg_name(name, width, nfrac)

    # make the final output files
    file_name = dest_folder + package_name + ".sv"
    dest_file = open(file_name, 'w')

    print(package_name)

    # first decalaration lines
    dest_file.write("// Width: " + str(width) + "\n// NFRAC: " + str(nfrac) + "\n")
    dest_file.write("package " + str(package_name) + ';\n\n')
    dest_file.write(f"localparam logic signed [{str(width-1)}:0] weights [{str(input_size)}][{str(output_size)}] = \'{{ \n")

    # convert weights (from floats) into binary
    for i in range(input_size):
        dest_file.write("{")
        for j in range(output_size):
            flt_num = float(src_weights_file.readline())
                        
            if (flt_num > max_val):
                flt_num = max_val
            elif (flt_num < min_val):
                flt_num = min_val

            binary_num = conv_to_str(flt_num, width, nfrac).replace('-', '')

            # Overflow checking, now handled by converting extreme values to max/min values
            if (len(binary_num) < width):
                print("ERROR: binary number too large")
                print("binary number: " + binary_num)
                print("float number: " + str(flt_num))
                print("width: " + str(width))
                print("binary number length: " + str(len(binary_num)))
                print("from file: " + weights_file)
                raise ValueError("Binary number width inconsistent")


            dest_file.write(str(width) + "\'b" + binary_num)
            if (j != output_size - 1):
                dest_file.write(", ")
            else:
                dest_file.write("}")

        if (i != (input_size-1)): # no comma for last term
            # OPTIONAL: adds floating number as comment as reference
            dest_file.write(", " + "\n")
        else:
            dest_file.write("\n")                
    
    # switch to declaring biases
    dest_file.write("};\n\n")
    dest_file.write("localparam logic signed ["+ str(width-1) + ":0] bias [" + str(output_size) + "] = \'{\n")
    
    # convert bias (from floats) into binary
    for i in range(output_size):
        flt_num = float(src_biases_file.readline())
        
        # e.g. write line "17'b01011011011111001,"
        dest_file.write(str(width) + "\'b" + conv_to_str(flt_num, width, nfrac).replace('-', ''))
        
        if (i != (output_size-1)): # no comma for last term
            # OPTIONAL: adds floating number as comment as reference
            dest_file.write(",  // " + str(flt_num) + "\n")
        else:
            dest_file.write("   // " + str(flt_num) + "\n")
        
        
    # finish off declarations
    dest_file.write("};\nendpackage")
    
    
    dest_file.close()
    src_biases_file.close()
    src_weights_file.close()

In [6]:
def generate_gru_pkg_file(width, nfrac, input_size, output_size):

    # input size for each gru internal gate is input_size + h, which is input_size + output_size
    generate_dense_pkg_file("candidate_gate", "./weights/gru_weights_biases_pkgs/candidate_gate_weights_biases_pkgs/", width, nfrac, "candidate_gate_weights.txt", "candidate_gate_biases.txt", input_size + output_size, output_size)
    generate_dense_pkg_file("reset_gate", "./weights/gru_weights_biases_pkgs/reset_gate_weights_biases_pkgs/", width, nfrac, "reset_gate_weights.txt", "reset_gate_biases.txt", input_size + output_size, output_size)
    generate_dense_pkg_file("update_gate", "./weights/gru_weights_biases_pkgs/update_gate_weights_biases_pkgs/", width, nfrac, "update_gate_weights.txt", "update_gate_biases.txt", input_size + output_size, output_size)

In [ ]:
# Weight generating file specific to ftag model.

# # Set width and fractional bits you want to generate
# width = 10
# nfrac = 5

# # run function
# generate_dense_pkg_file("dense_0", width, nfrac, "dense_0_weights.txt", "dense_0_biases.txt", NUM_WEIGHTS_DENSE_0, NUM_BIAS_DENSE_0)
# generate_dense_pkg_file("dense_1", width, nfrac, "dense_1_weights.txt", "dense_1_biases.txt", NUM_WEIGHTS_DENSE_1, NUM_BIAS_DENSE_1)

# generate set of files for bit width 4-32
for i in range(4,33):
    width = i
    nfrac = (i+1) // 2
    
    # generate_input_bin_file(width, nfrac)
    generate_dense_pkg_file("dense_0", "./weights/dense_0_weights_biases_pkgs/", width, nfrac, "dense_0_weights.txt", "dense_0_biases.txt", INPUT_SIZE_DENSE_0, OUTPUT_SIZE_DENSE_0)
    generate_dense_pkg_file("dense_1", "./weights/dense_1_weights_biases_pkgs/", width, nfrac, "dense_1_weights.txt", "dense_1_biases.txt", INPUT_SIZE_DENSE_1, OUTPUT_SIZE_DENSE_1)
    generate_gru_pkg_file(width, nfrac, INPUT_SIZE_GRU, OUTPUT_SIZE_GRU)
    # output layer
    generate_dense_pkg_file("output", "./weights/output_weights_biases_pkgs/", width, nfrac, "output_softmax_weights.txt", "output_softmax_biases.txt", INPUT_SIZE_OUT_DENSE, OUTPUT_SIZE_OUT_DENSE)

Max value: 1.75
Min value: -2.0
dense_0_4_2
Max value: 1.75
Min value: -2.0
dense_1_4_2
Max value: 1.75
Min value: -2.0
candidate_gate_4_2
Max value: 1.75
Min value: -2.0
reset_gate_4_2
Max value: 1.75
Min value: -2.0
update_gate_4_2
Max value: 1.75
Min value: -2.0
output_4_2
Max value: 1.875
Min value: -2.0
dense_0_5_2
Max value: 1.875
Min value: -2.0
dense_1_5_2
Max value: 1.875
Min value: -2.0
candidate_gate_5_2
Max value: 1.875
Min value: -2.0
reset_gate_5_2
Max value: 1.875
Min value: -2.0
update_gate_5_2
Max value: 1.875
Min value: -2.0
output_5_2
Max value: 3.875
Min value: -4.0
dense_0_6_3
Max value: 3.875
Min value: -4.0
dense_1_6_3
Max value: 3.875
Min value: -4.0
candidate_gate_6_3
Max value: 3.875
Min value: -4.0
reset_gate_6_3
Max value: 3.875
Min value: -4.0
update_gate_6_3
Max value: 3.875
Min value: -4.0
output_6_3
Max value: 3.9375
Min value: -4.0
dense_0_7_3
Max value: 3.9375
Min value: -4.0
dense_1_7_3
Max value: 3.9375
Min value: -4.0
candidate_gate_7_3
Max value: 3

In [ ]:
width = 16
nfrac = 6
generate_gru_pkg_file(width, nfrac, INPUT_SIZE_GRU, OUTPUT_SIZE_GRU)
generate_dense_pkg_file("dense_0", "./weights/dense_0_weights_biases_pkgs/", width, nfrac, "dense_0_weights.txt", "dense_0_biases.txt", INPUT_SIZE_DENSE_0, OUTPUT_SIZE_DENSE_0)
generate_dense_pkg_file("dense_1", "./weights/dense_1_weights_biases_pkgs/", width, nfrac, "dense_1_weights.txt", "dense_1_biases.txt", INPUT_SIZE_DENSE_1, OUTPUT_SIZE_DENSE_1)
# output layer
generate_dense_pkg_file("output", "./weights/output_weights_biases_pkgs/", width, nfrac, "output_softmax_weights.txt", "output_softmax_biases.txt", INPUT_SIZE_OUT_DENSE, OUTPUT_SIZE_OUT_DENSE)